# Procesamiento del lenguaje natural con modelos preentrenados

En el ejemplo anterior vimos cómo utilizar bibliotecas como NLTK y TextBlob para procesar texto. Aunque son eficientes y fáciles de usar, no siempre son las mejores opciones para trabajar con lenguas distintas del inglés.

Los modelos preentrenados se han entrenado con grandes cantidades de datos y pueden utilizarse para procesar textos en varios idiomas. Entrenarlos requiere más datos y tiempo de cálculo, pero una vez entrenados pueden ser eficientes.

Además, su entrenamiento requiere menos intervención manual y facilita su uso en distintas lenguas.

En esta práctica veremos cómo utilizar modelos preentrenados para procesar textos en diferentes idiomas.

Utilizaremos modelos de la biblioteca `flair`, aunque también podríamos emplear otras colecciones, como `spaCy`.

## Preparación del entorno

Primero instalaremos la biblioteca `flair`.

In [1]:
!pip install flair

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 8.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 7.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 6.4 MB/s eta 0:00:00a 0:00:01
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993225 sha256=6cf8e98f643d99c407c4e0a815b6c10055845867384fd10347d6e55b038797ef
  Stored in directory: /home/carles/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
  Created wheel for pptree: filename=pptree-3.1-py3-none-any.whl si

## Tokenización

Para tokenizar con spaCy, creamos una instancia `Sentence` con el texto que queremos procesar. La utilizaremos en muchas de las operaciones siguientes.

In [36]:
from flair.data import Sentence

sentence = Sentence('Els dilluns a la tarda són molt llargs.')

sentence

Sentence[9]: "Els dilluns a la tarda són molt llargs."

La salida indica que la frase contiene nueve tokens.

In [37]:
for token in sentence:
    print(token)

Token[0]: "Els"
Token[1]: "dilluns"
Token[2]: "a"
Token[3]: "la"
Token[4]: "tarda"
Token[5]: "són"
Token[6]: "molt"
Token[7]: "llargs"
Token[8]: "."


Para acceder a los tokens podemos recorrer el objeto `Sentence` como si fuera una lista.

In [38]:
for token in sentence:
    print(token.text)

Els
dilluns
a
la
tarda
són
molt
llargs
.


Si solo necesitamos los tokens, podemos obtener el atributo `text` de cada uno.

In [34]:
from flair.nn import Classifier

tagger = Classifier.load('pos-multi')
tagger.predict(sentence)

# Mostrem els tags de cada paraula
for token in sentence:
    print(token)

2025-02-17 23:45:54,740 SequenceTagger predicts: Dictionary with 17 tags: NOUN, PUNCT, ADP, VERB, ADJ, DET, PROPN, ADV, PRON, AUX, CCONJ, NUM, SCONJ, PART, X, SYM, INTJ
Token[0]: "Els" → DET (0.9979)
Token[1]: "dilluns" → NOUN (0.9716)
Token[2]: "a" → ADP (0.9995)
Token[3]: "la" → DET (0.9998)
Token[4]: "tarda" → NOUN (0.9897)
Token[5]: "són" → AUX (0.4685)
Token[6]: "molt" → ADV (0.6738)
Token[7]: "llargs" → ADJ (0.5771)
Token[8]: "." → PUNCT (1.0000)


Una de las funciones más interesantes de spaCy permite obtener la categoría gramatical de cada palabra. Para ello, crearemos un clasificador de texto con el modelo preentrenado `pos-multi`.

### Tokenización de frases

A veces también es necesario dividir el texto en frases. Para hacerlo, crearemos un `Splitter` de tipo `SegtokSentenceSplitter` y utilizaremos el método `split`.

Flair admite otros segmentadores. Utilizaremos `SegtokSentenceSplitter` porque es sencillo y eficiente.

In [35]:
from flair.splitter import SegtokSentenceSplitter

# Tokenitzem el text en frases

splitter = SegtokSentenceSplitter()

text = "El Barça és el millor equip del món. De vegades."
sentences = splitter.split(text)

for sentence in sentences:
    print(sentence)

Sentence[9]: "El Barça és el millor equip del món."
Sentence[3]: "De vegades."


## Palabras vacías y signos de puntuación

Las *stopwords* son palabras que aportan poca información para una tarea concreta. Por ejemplo, en la clasificación de textos normalmente no ayudan a decidir la categoría.

Por eso, en muchos casos conviene eliminarlas antes de aplicar técnicas de procesamiento del lenguaje natural.

Utilizaremos la lista de *stopwords* de NLTK.

In [ ]:
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stop = stopwords.words('catalan')

[nltk_data] Downloading package stopwords to /home/carles/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [44]:

text = "El Barça és el millor equip del món. De vegades."
sentence = Sentence(text)
tagger.predict(sentence)

tokens = [token.text for token in sentence if token.text not in stop]
tokens

sentence

Sentence[12]: "El Barça és el millor equip del món. De vegades." → ["El"/DET, "Barça"/PROPN, "és"/AUX, "el"/DET, "millor"/NOUN, "equip"/NOUN, "del"/X, "món"/NOUN, "."/PUNCT, "De"/ADP, "vegades"/PROPN, "."/PUNCT]

También podemos observar que, junto con las palabras vacías, se eliminan los signos de puntuación.

In [43]:
tokens = [token.text for token in sentence if token.text not in stop and not token.tag == 'PUNCT']
tokens

['El', 'Barça', 'millor', 'equip', 'món', 'De', 'vegades']

## Lematización y *stemming*

La lematización convierte una palabra a su forma base. Por ejemplo, *está* pasa a *estar*. El *stemming*, en cambio, elimina los afijos: *está* podría convertirse en *est*.

Ambas técnicas reducen el vocabulario y, por tanto, la dimensionalidad de los vectores de palabras. Esto puede mejorar el rendimiento de los modelos.

spaCy no incluye un método de *stemming*, pero sí permite lematizar. Para ello, consultamos el atributo `lemma_` de cada palabra.

spaCy no incluye *stemming* porque la lematización suele ser más precisa. La biblioteca está orientada a tareas de producción, donde la precisión puede ser más importante que la velocidad.

In [46]:
# Lematitzem i stemitzem un text

text = "Series un bon cantant, si no fora per la veu."
doc = nlp(text)

# Lematitzem
lemmas = [token.lemma_ for token in doc]

lemmas

['seria',
 'un',
 'bo',
 'cantant',
 ',',
 'si',
 'no',
 'fora',
 'per',
 'el',
 'veu',
 '.']

## Representación del texto

Después de preprocesar el texto, debemos representarlo en un formato que el ordenador pueda procesar.

spaCy está orientado a tareas de producción e incluye métodos de representación habituales, especialmente representaciones basadas en vectores de características.

### *Embeddings*

spaCy permite obtener los *embeddings* de documentos y palabras. Son vectores de características que representan un documento o una palabra.

Se pueden utilizar en tareas de clasificación y agrupamiento de textos, entre otras.

Como utilizamos el modelo pequeño (`ca_core_news_sm`), los *embeddings* son menos precisos y solo están disponibles a nivel de documento, no de palabra. Aun así, pueden servir para tareas sencillas.

Si necesitamos vectores más precisos, podemos utilizar los modelos mediano (`ca_core_news_md`) o grande (`ca_core_news_lg`).

In [47]:
doc.vector

array([ 1.7305475 , -0.12840228, -0.5396843 , -0.83717424, -1.3100742 ,
        0.17650956, -0.1620681 ,  0.48682687,  0.06764957,  0.18184917,
        0.8215378 , -0.40531054,  0.25735107,  0.93934184, -0.06517167,
        0.04939266, -0.4045429 ,  0.24763483,  1.0905582 ,  0.6970573 ,
       -0.31526443,  1.4649282 , -0.23816238, -0.4865251 , -0.09747415,
       -0.28767216,  0.5814107 , -0.7125825 ,  0.49832496,  0.8188111 ,
       -0.15080123,  0.1994148 ,  0.2723398 ,  0.8346703 , -0.7402689 ,
       -1.4915727 , -0.9905634 ,  0.42298186,  0.6075589 ,  0.29053253,
       -0.00331459, -1.3474356 , -0.58441734, -0.7741163 ,  0.05676066,
       -0.17020352,  0.40141544,  0.6308312 , -0.40938064,  0.6958816 ,
        0.3174107 ,  1.168798  , -0.04763202,  0.19581653,  0.08472364,
       -0.01598583,  0.00706673, -0.8146202 , -0.65326816, -0.6593003 ,
       -1.0186236 , -1.1242762 ,  0.1646251 ,  0.07394195, -0.7488492 ,
       -0.5852267 , -0.8377841 , -0.47709122,  0.6825135 ,  0.55

## Similitud entre textos

A partir de los *embeddings*, spaCy permite calcular la similitud entre documentos o palabras. Se calcula mediante la similitud del coseno entre los vectores de características.

Esta funcionalidad puede utilizarse para buscar sinónimos, crear sistemas de recomendación, etc.

Para obtener vectores de palabras y calcular su similitud, necesitamos cargar el modelo mediano (`ca_core_news_md`) o el grande (`ca_core_news_lg`).

### Ejemplo de sinónimos

In [54]:
# Carreguem un model de paraules

spacy.cli.download("ca_core_news_md")
nlp = spacy.load("ca_core_news_md")

  Using cached https://github.com/explosion/spacy-models/releases/download/ca_core_news_md-3.8.0/ca_core_news_md-3.8.0-py3-none-any.whl (49.2 MB)



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


✔ Download and installation successful
You can now load the package via spacy.load('ca_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [62]:
def most_similar(word, topn=5):
    ms = nlp.vocab.vectors.most_similar(
        nlp(word).vector.reshape(1, nlp(word).vector.shape[0]), n=topn
    )
    words = [nlp.vocab.strings[w] for w in ms[0][0]]
    distances = ms[2]
    return words, distances


doc = nlp("Els dilluns a la tarda són molt llargs.")

for token in doc:
    print(token.text, most_similar(token.text, topn=3))

Els (['2.-Els', 'Fels', 'Algúns'], array([[1.    , 0.7737, 0.7637]], dtype=float32))
dilluns (['dillun', '-dimarts', 'dimartts'], array([[1.    , 0.9662, 0.9593]], dtype=float32))
a (['a', 'al', 'podrar'], array([[1.    , 0.343 , 0.3332]], dtype=float32))
la (['deLa', 'PLa', 'ceva'], array([[1.    , 0.708 , 0.6945]], dtype=float32))
tarda (['latarda', 'vespri', 'matí-migdia'], array([[1.    , 0.8403, 0.8243]], dtype=float32))
són (['són.', 'Sóu', 'Arien'], array([[1.    , 0.8283, 0.7558]], dtype=float32))
molt (['skolt', '-bastant', 'cagaire'], array([[1.    , 0.879 , 0.8005]], dtype=float32))
llargs (['llargs-', 'hurts', 'camesllargues'], array([[1.    , 0.7822, 0.7382]], dtype=float32))
. (['.', '</s>', 'i'], array([[1.    , 0.9258, 0.5367]], dtype=float32))


# Conclusiones

En esta práctica hemos visto cómo preprocesar el texto y representarlo en un formato que pueda procesar un ordenador. También hemos utilizado algunas funciones de TextBlob y de bibliotecas relacionadas.

TextBlob se basa en NLTK, una biblioteca fundamentada en reglas. Como hemos visto en la teoría, estas herramientas no siempre funcionan bien. En la siguiente práctica utilizaremos herramientas basadas en redes neuronales para preprocesar y clasificar textos.